In [ ]:

import kagglehub
brendan45774_test_file_path = kagglehub.dataset_download('brendan45774/test-file')

print('Data source import complete.')


In [ ]:


import numpy as np
import pandas as pd



import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



In [ ]:
### I will upload the dataset which is training and testing from my local system, due to some issues which i am facing while loading the data from kaggle...

In [ ]:
titanic_train_df = pd.read_csv('/kaggle/input/titanic/train.csv')

In [ ]:
titanic_test_df = pd.read_csv('/kaggle/input/titanic/test.csv')

In [ ]:
titanic_train_df.shape

In [ ]:
titanic_test_df.shape

In [ ]:
titanic_train_df.info()

In [ ]:
titanic_test_df.info()

In [ ]:
## So we have total of 891 data in our training data, which contains floating, int and object type values...
## For Now we will consider our training data only

In [ ]:
## check for any null values in the data

titanic_train_df.isnull().sum()

In [ ]:
### So  Age, Cabin and Embarked Column contains null values, Age is an Int Column, Cabin and embarked are categorical columns...

In [ ]:
titanic_train_df['Cabin'].value_counts()

In [ ]:
titanic_train_df['Embarked'].value_counts()

In [ ]:
## We will going to plot the features so that we can know whether the distributions are Normal or skewed and based upon that we
## will be going to Impute the values in place of NAN or null values in our dataset for the Int column which is 'Age'.... for Categorical Data Most Frequent
## Value is Imputed...

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.rc('font', size = 12)
plt.rc('axes', labelsize = 12, titlesize = 14)
plt.rc('xtick', labelsize = 12)
plt.rc('ytick', labelsize = 12)
plt.rc('legend', fontsize = 13)

titanic_train_df.hist(bins = 40, figsize=(10, 8))
plt.tight_layout()
plt.show()

In [ ]:
## As we can see Age is Non-normally distributed so we will use "median" as an imputation for null values...

In [ ]:
## But Before doing all this, we have to first separate the target column from our training dataset...

In [ ]:
X_train = titanic_train_df.drop('Survived', axis = 1)

y_train = titanic_train_df['Survived'].copy()

In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler



In [ ]:
## Now For the categorical Pipeline what we will do is we will first list down all the categorical variables to see which ones are:
## ordinal type and which ones are Nominal TYpes then apply the Encoding Starategy, Imputation strategy...

In [ ]:
cat_cols = X_train.select_dtypes(include = 'object').columns.tolist()

In [ ]:
cat_cols

In [ ]:
X_train.select_dtypes(include='object').head(3)

In [ ]:
## As we can see not all the categorical columns are much of an importance with regards to prediction for the survivors so we will handle them
## One by one

**Name-> Not Directly Useful but we could extract titles from them and that can be useful, Sex -> this column is really important , ticket -> ticket column is largely random and will not help in predicting the survivors, cabin -> can also be used , embarked -> useful column**

In [ ]:
X_train['Name_Title'] = X_train['Name'].str.extract(r' ([A-Za-z]+)\.', expand = False).replace(['Mille', 'Ms'], 'Miss').replace(['Mmr'], 'Mrs').replace(
        ["Dr", "Rev", "Col", "Major", "Capt", "Don", "Sir",
         "Jonkheer", "Lady", "Countess", "Dona"],
        "Rare"
    )

X_train['cabin_first_letter'] = X_train['Cabin'].str[0]


### Extracting the titles from the Name Column and creating a new column with title name values...
### Extracting the cabin column's first letter...

In [ ]:
## We will drop the ticket column and other modified columns bcoz it is not much of an importance in terms of predicting about the Survivors...

In [ ]:
X_train = X_train.drop(['Name', 'Ticket', 'Cabin'], axis = 1)

In [ ]:
X_train.head(3)

In [ ]:
X_train = X_train.drop(['PassengerId'], axis = 1)

In [ ]:
## Now That we have succussfully Take out the important stuffs from our columns, we will proceed towards building ppelines for the desired work..

In [ ]:
## Since Sex Column has two values which are ordinal ones so we will apply Ordinal Encoders on them,
## and for other columns we will apply One-Hot Encoding...Imputation of Most Freaquent Data will be applied to NAN or null values...

In [ ]:
ordinal_cat_col = ['Sex']



In [ ]:
from sklearn.preprocessing import OrdinalEncoder
ordinal_cat_pipeline = Pipeline(steps=[
    ('ordinal_encode', OrdinalEncoder())
])

In [ ]:
one_hot_encoder_cat_cols = ['Embarked', 'Name_Title', 'cabin_first_letter']

In [ ]:
from sklearn.preprocessing import OneHotEncoder

one_hot_cat_pipeline = Pipeline(steps=[
    ('Imputation', SimpleImputer(strategy='most_frequent')),
    ('one_hot_encode', OneHotEncoder(handle_unknown='ignore'))
])

In [ ]:
num_cols = X_train.select_dtypes(include = np.number).columns.tolist()

num_cols

In [ ]:
num_pipeline = Pipeline(steps = [
    ('Imputation', SimpleImputer(strategy = 'median')),
    ('Scaling', StandardScaler())
])

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num_transform', num_pipeline, num_cols),
        ('ordinal_transform', ordinal_cat_pipeline, ordinal_cat_col),
        ('one-hot_transform', one_hot_cat_pipeline, one_hot_encoder_cat_cols)
    ]
)

In [ ]:
preprocessor

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(random_state=42)

In [ ]:
model = Pipeline(
    steps = [
        ('preprocessing', preprocessor),
        ('classifier', rf_clf)
    ]
)

In [ ]:
model.fit(X_train, y_train)

**Now we will check the score on training set**

In [ ]:
train_score = model.score(X_train, y_train)

train_score

In [ ]:
## The above score is coming out to be too good to be true, which might be the high possibility that our model is Overfitting the data...

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
Cv_scores = cross_val_score(model, X_train, y_train, cv = 3, scoring = "accuracy")

In [ ]:
Cv_scores

**Now As we  can clearly see our model was extremely overfitting when we were direclty using the scores but the actual numbers we can see with Cross Validation Scores**

In [ ]:
### We can Use a way to see about how our classifier has predicted the values on input set, how much of the values were predicted as 0 and
### how much of them are predicted as 1 either correct or incorrect...

In [ ]:
from sklearn.metrics import confusion_matrix

from sklearn.model_selection import cross_val_predict

In [ ]:
y_pred_train = cross_val_predict(model, X_train, y_train, cv = 3)

In [ ]:
confusion_matrix(y_train, y_pred_train)

In [ ]:
### Each Row Represent actual class and each column represents the predicted class...
## 471 True Negative, 78 False Positive, 243 True Positive, 99 False Negative...

In [ ]:
y_train.value_counts()

In [ ]:
## So the first row in the Confusion Matrix Corresponds to the '0' class which is not survived, and the second row corresponds to the '1' class
## which means he/she survived...

In [ ]:
## 471 is true 0's , 78 is false 1's  ;; 99 is false 0's and 243 means true 1's, ...
## based upon these we can calculate  the precision and recall scores as well...

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score    ## f1 score is harmonic mean of precion and recall...


In [ ]:
precision_score(y_train, y_pred_train)

In [ ]:
recall_score(y_train, y_pred_train)

In [ ]:
f1_score(y_train, y_pred_train)

In [ ]:
## the regular mean treats all values equally but a harmonic mean which is f1 score gives more weight to the low values...
## so the classifier will have a high f1 score only when both the precision and recalls are high...

In [ ]:
## we can go with either precision or recall because increasing one decreases another and this

## is basically called a precision/recall trade off
## which one you target for depends upon context to context...

**I would like my classifier to have higher preiscion so i have to compromise a little with the recall bcoz if the precison goes up recall goes down....So basically the classifier makes decisions about any instance which it come acorss to be of positive class or negative class based upon the decision function score then it basically compares the score with the threshold value if it is greater than threshold value then it is a positive class and if it is lower than it is of a negative class**

In [ ]:
## But scikit learn does not directly provides any measure to set the threshold value, so we will use the scores to compute
## the index at some precision value that we want say 85%, then we will set the threshold value from there...

In [ ]:
y_scores_rf = cross_val_predict(model, X_train, y_train,cv = 3, method = 'predict_proba')

In [ ]:
## Now lets look at the estimated probabilities for the first 3 instances...

y_scores_rf[:3]

In [ ]:
y_train[:3]

In [ ]:
## this shows us that our classifier is 90% estimated that our first instance is 0 which is correct,
## for the second instance it is 99% estimated that our second instance is 1,
## for the third instance it has predicted the estimated probability 75% for 0 class and 25% for 1... for the third instance it is wrong...

In [ ]:
### So i want my classifier to predict class '1' which is basically people who survived with 85% precision currently my
## precision is at 75% approx...

In [ ]:
y_pred_survived_scores = y_scores_rf[:,1]  ## bcoz i am interested in class '1' which means people who survived...

In [ ]:
y_pred_survived_scores[:3]

In [ ]:
from sklearn.metrics import precision_recall_curve

precision_forest, recall_forest, thresholds_forest = precision_recall_curve(y_train, y_pred_survived_scores)


In [ ]:
## now we will try to find the max id for atleast which the precision is 85%,

id_for_85_precision = (precision_forest >= 0.85).argmax()

threshold_for_85_precision = thresholds_forest[id_for_85_precision]

In [ ]:
threshold_for_85_precision

In [ ]:
## so we have got our threshold for 85% precision, now we can compute y_pred_survived_scores for 85% precision,

In [ ]:
y_pred_scores_85_precision = y_pred_survived_scores >= threshold_for_85_precision

In [ ]:
precision_score(y_train, y_pred_scores_85_precision)

In [ ]:
recall_score(y_train, y_pred_scores_85_precision)

**Notice that our recall got lesser as compared to its previous value**

In [ ]:
## Since in our data our Positive class which is 1 is somewhat closer to the negative class which is 0,
## so we will use Roc-Auc Score to compare between two classifier...but we can also use Precision-Recall Curve as well

**Now We will create a new Model with SGD classifier now start to see their precision, score, recall score, and how we can improve that as well**

In [ ]:
from sklearn.linear_model import SGDClassifier

sgd_clf = SGDClassifier(random_state=42)

In [ ]:
sgd_clf_model = Pipeline(steps = [
    ('preprocessing', preprocessor),
    ('SGD_Classifier', sgd_clf)
])

In [ ]:
sgd_clf_model.fit(X_train, y_train)

In [ ]:
sgd_training_score = sgd_clf_model.score(X_train, y_train)

In [ ]:
sgd_training_score

In [ ]:
## This score is less as compared to our forestclassifier but not much difference in scores...

In [ ]:
cross_val_score_SGD = cross_val_score(sgd_clf_model, X_train, y_train, cv = 3,
                                      scoring='accuracy')

In [ ]:
cross_val_score_SGD

In [ ]:
y_predict_sgd = cross_val_predict(sgd_clf_model, X_train, y_train, cv=3)

In [ ]:
y_predict_sgd[:3]

In [ ]:
confusion_matrix(y_train, y_predict_sgd)

In [ ]:
precision_score(y_train, y_predict_sgd)

In [ ]:
recall_score(y_train, y_predict_sgd)

In [ ]:
### For SGD Classifier our recall rate is 78% and precision is 73%,


In [ ]:
## We Will now compute the y_scores from the decision_function method based upon which the classifier determines whether to classify the
## class as Positive or Negative....

In [ ]:
from sklearn.metrics import roc_auc_score

y_scores_SGD = cross_val_predict(sgd_clf_model, X_train, y_train, cv = 3,
                                 method = 'decision_function')

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_train, y_scores_SGD)

In [ ]:
id_for_85_pre_SGD = (precisions >= 0.85).argmax()

threshold_for_85_pre_SGD = thresholds[id_for_85_pre_SGD]

threshold_for_85_pre_SGD

In [ ]:
y_train_pred_85 = y_scores_SGD >= threshold_for_85_pre_SGD

In [ ]:
precision_score(y_train, y_train_pred_85)

In [ ]:
recall_score(y_train, y_train_pred_85)

**Notice that the recall has significantly reduced which is not good for the classifier, always try to balance between these two values**

**WE WILL CHECK THE ROC_AUC_SCORES Which will give us a comparison about which classifier is performing good**


In [ ]:
## ROC_AUC Score on SGD Classifier...

roc_auc_score(y_train, y_scores_SGD)

In [ ]:
## ROC_AUC Score on Random Forest Classifier

roc_auc_score(y_train, y_pred_survived_scores)

In [ ]:
## ROC AUC Score on Scores with 85% precision RAndom Forest classifier...

roc_auc_score(y_train, y_pred_scores_85_precision)

In [ ]:
### ROC AUC Score on Scores with 85% precision on SGD CLassifier

roc_auc_score(y_train, y_train_pred_85)

In [ ]:
### NOW Lets make a precision Recall Curve

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.plot(thresholds, precisions[:-1], "b--", label="Precision", linewidth=2)
plt.plot(thresholds, recalls[:-1], "g-", label="Recall", linewidth=2)

best_threshold = 3.35   ## this is the threshold for which we were getting 85% precision
plt.axvline(x=best_threshold, color="k", linestyle="dotted", linewidth=2, label=f"Threshold = {best_threshold:.2f}")


plt.xlim(thresholds.min(), thresholds.max())
plt.ylim(0, 1)


plt.title("Precision and Recall vs Decision Threshold", fontsize=14, pad=15)
plt.xlabel("Threshold", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.legend(loc="best")
plt.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
## This Graph shows us that in our case our recall value is coming out to be very low for a precision of 85%,
## SO we should instead choose some other threshold for which we should not overcompensate for our recall value,
## For that we should aim for threshold value some where between 0 and 2

| Metric            | **Random Forest** | **SGD Classifier** | Comment                                        |
| :---------------- | :---------------- | :----------------- | :--------------------------------------------- |
| Training accuracy | **98.4%**         | 81.2%              | RF seems overfitted (too high compared to CV). |
| Cross-val mean    | **0.80**          | **0.81**           | Both similar in generalization.                |
| Precision         | **0.757**         | **0.731**          | RF slightly better.                            |
| Recall            | **0.711**         | **0.787**          | SGD slightly better (catches more survivors).  |
| F1 score          | **0.733**         | **0.759**          | Similar trade-off.                             |
| ROC-AUC           | **0.848**         | **0.831**          | RF again slightly better.                      |


In [ ]:
## So random forest is perform slightly better than SGD classifiers...bcoz it is giving us slightly better precision scores, so we will go with Random Classifier...

In [ ]:
## But BEfore testing it on unseen data, we can improve it first or should i say fine tune it...

In [ ]:
## We will use Randomized Search CV to find the best settings in this scenario..

In [ ]:
RandomForestClassifier()

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'classifier__n_estimators' : [100, 200, 300],
    'classifier__max_depth' : [5, 10, 15, None],
    'classifier__min_samples_split' : [2, 4, 10],
    'classifier__min_samples_leaf' : [1, 2, 4],
    'classifier__max_features' : ['log2', 'sqrt']
}

In [ ]:
Random_search = RandomizedSearchCV(model, param_distributions=param_dist, n_iter=20,
                                   scoring = 'roc_auc', cv = 3, random_state=42, n_jobs= -1)

Random_search.fit(X_train, y_train)

In [ ]:
### Now we will see what our best_estimators are...

In [ ]:
Random_search.best_estimator_

In [ ]:
Random_search.best_params_

In [ ]:
Random_search.best_score_

In [ ]:
## Our Last Score for Random Forest Classifier with random selection of parameters for X_train and y_train  was 0.84

In [ ]:
improved_cv_scores = cross_val_score(Random_search.best_estimator_, X_train, y_train, cv=3,
                                     scoring = 'roc_auc')

In [ ]:
improved_cv_scores

In [ ]:
## As we can see our roc_auc scores are pretty much improved from our last scores...

In [ ]:
## Now we can Ready our final classifier with best parameters to test it on the final testing data....

In [ ]:
tested = pd.read_csv('/kaggle/input/test-file/tested.csv')

In [ ]:
tested.shape

In [ ]:
tested.head(3)

In [ ]:
## we will just have to clean the data and put it in the pipeline with the best estimators of random classifier

In [ ]:
X_test = tested.drop(['Survived'], axis = 1)

y_test = tested['Survived'].copy()

In [ ]:
X_test.shape

In [ ]:
y_test.shape

In [ ]:
X_test['Name_Title'] = X_test['Name'].str.extract(r' ([A-Za-z]+)\.', expand = False).replace(['Mille', 'Ms'], 'Miss').replace(['Mmr'], 'Mrs').replace(
        ["Dr", "Rev", "Col", "Major", "Capt", "Don", "Sir",
         "Jonkheer", "Lady", "Countess", "Dona"],
        "Rare"
    )

X_test['cabin_first_letter'] = X_test['Cabin'].str[0]


### Extracting the titles from the Name Column and creating a new column with title name values...
### Extracting the cabin column's first letter...

In [ ]:
X_test.drop(['Name', 'Cabin', 'PassengerId', 'Ticket'], inplace = True, axis = 1)

In [ ]:
X_test.head(3)

In [ ]:
## Now we are all set...

In [ ]:
best_model = Random_search.best_estimator_

In [ ]:
best_model.fit(X_test, y_test)

In [ ]:
best_model

In [ ]:
best_model.score(X_test, y_test)

In [ ]:
Cv_scores_for_best_model = cross_val_score(best_model, X_test, y_test, cv = 3, scoring = 'accuracy')

Cv_scores_for_best_model

In [ ]:
y_pred_on_test_set = cross_val_predict(best_model, X_test, y_test, cv=3)

In [ ]:
confusion_matrix(y_test, y_pred_on_test_set)

In [ ]:
precision_score(y_test, y_pred_on_test_set)

In [ ]:
recall_score(y_test, y_pred_on_test_set)